# Trading Decision Maker — News + Macro + AI Edition
**Workflow:** Screen tickers → Score fundamentals & technicals → Collect RSS news & FRED macro → Composite 4-factor score → Rule-based reports → 🤖 Ollama AI narratives

> **What's in this notebook:**
> - 📊 Fundamental screen: ROE, D/E, P/E, Piotroski F-Score, Altman Z-Score
> - 📈 Technical screen: 50/200 DMA, RSI, 52-week high distance
> - 📰 RSS news collector (Reuters, Yahoo Finance, FT, WSJ) via `feedparser` — no API key
> - 🗞️ Per-ticker company news via `yfinance` with keyword sentiment scoring
> - 🏛️ FRED macro score: Fed rate, CPI, unemployment, yield spread, VIX, GDP
> - 📊 4-factor composite score: 40% Fundamental + 25% Technical + 20% News + 15% Macro
> - 🤖 **Ollama AI narratives**: local LLM generates a structured one-page investment summary per ticker
> - 🏆 Full decision report and summary dashboard

> **One-time setup:** Run Cell 2 once, then comment it out.

In [37]:
# ── Run once, then comment out ───────────────────────────────────────────────
# !pip install yfinance pandas feedparser fredapi ollama
!pip install feedparser
!pip install fredapi

---
## ⚙️ CELL 3 — YOUR CONFIGURATION (edit only this cell)
---

In [38]:
import datetime
from dataclasses import dataclass, field
from pathlib import Path
import pandas as pd
import yfinance as yf

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1 — TICKER LIST                                                   ║
# ║  Replace the tickers below with the stocks you want to screen.          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

TICKERS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "GOOGL",
    "META",
    "JPM",
    "TSLA",
]

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2 — FRED API KEY                                                  ║
# ║  Get a free key at: https://fred.stlouisfed.org/docs/api/api_key.html  ║
# ╚══════════════════════════════════════════════════════════════════════════╝

FRED_API_KEY = "5758685d1d96b52dedd08d7933375085"   # ← your key here

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 — ANALYSIS DATE (defaults to today)                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

ANALYSIS_DATE = datetime.date.today().strftime("%Y-%m-%d")

# ──────────────────────────────────────────────────────────────────────────
# OPTIONAL — Screening thresholds
# ──────────────────────────────────────────────────────────────────────────
MIN_ROE     = 0.10   # minimum Return on Equity  (10%)
MAX_DE      = 2.0    # maximum Debt-to-Equity ratio
MAX_PE      = 30.0   # maximum Price-to-Earnings ratio
MIN_F_SCORE = 6      # minimum Piotroski F-Score (out of 9)
MIN_Z_SCORE = 1.8    # minimum Altman Z-Score

# ── Composite score weights ───────────────────────────────────────────────
W_FUNDAMENTAL = 0.40
W_TECHNICAL   = 0.25
W_NEWS        = 0.20
W_MACRO       = 0.15

# ── Confirmation ──────────────────────────────────────────────────────────
print("Configuration loaded.")
print(f"  Analysis date : {ANALYSIS_DATE}")
print(f"  Tickers       : {TICKERS}")
print(f"  Thresholds    : ROE>={MIN_ROE:.0%}  D/E<={MAX_DE}  P/E<={MAX_PE}  F>={MIN_F_SCORE}  Z>={MIN_Z_SCORE}")
print(f"  Weights       : Fund={W_FUNDAMENTAL:.0%}  Tech={W_TECHNICAL:.0%}  News={W_NEWS:.0%}  Macro={W_MACRO:.0%}")
print(f"  FRED key      : {'set ✓' if FRED_API_KEY else 'NOT SET — macro score will be skipped'}")

Configuration loaded.
  Analysis date : 2026-06-23
  Tickers       : ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'JPM', 'TSLA']
  Thresholds    : ROE>=10%  D/E<=2.0  P/E<=30.0  F>=6  Z>=1.8
  Weights       : Fund=40%  Tech=25%  News=20%  Macro=15%
  FRED key      : set ✓


---
## Cell 4 — Fundamental screen helpers (no edits needed)
---

In [39]:
US_EXCHANGES = {"ASE","AMEX","BATS","NASDAQ","NCM","NGM","NMS","NYQ","NYSE","PCX"}

@dataclass
class FundamentalMetrics:
    ticker: str
    roe: float | None
    debt_to_equity: float | None
    pe_ratio: float | None
    piotroski_f_score: int
    altman_z_score: float | None
    fundamental_score: int
    fundamental_rating: str
    passed: bool

@dataclass
class TechnicalMetrics:
    ticker: str
    price: float | None
    ma_50: float | None
    ma_200: float | None
    rsi_14: float | None
    high_52w: float | None
    distance_from_52w_high: float | None
    technical_score: int
    technical_rating: str

def safe_div(n, d):
    return None if n is None or d in (None, 0) else n / d

def latest(stmt, row):
    if stmt is None or stmt.empty or row not in stmt.index: return None
    v = stmt.loc[row].dropna()
    return float(v.iloc[0]) if len(v) else None

def previous(stmt, row):
    if stmt is None or stmt.empty or row not in stmt.index: return None
    v = stmt.loc[row].dropna()
    return float(v.iloc[1]) if len(v) > 1 else None

def first_available(stmt, names):
    for n in names:
        v = latest(stmt, n)
        if v is not None: return v
    return None

def prior_available(stmt, names):
    for n in names:
        v = previous(stmt, n)
        if v is not None: return v
    return None

def is_us_stock(info):
    exchange   = str(info.get('exchange') or '').upper()
    full_ex    = str(info.get('fullExchangeName') or '').lower()
    country    = str(info.get('country') or '').lower()
    quote_type = str(info.get('quoteType') or '').upper()
    is_equity  = quote_type in {'EQUITY', ''}
    is_us      = country in {'united states','united states of america','usa','us',''}
    is_us_ex   = exchange in US_EXCHANGES or any(
        n in full_ex for n in ['nasdaq','nyse','american stock exchange','nyse arca','bats'])
    return is_equity and is_us and is_us_ex

def calc_piotroski(d):
    s    = 0
    roa  = safe_div(d["net_income"],       d["total_assets"])
    proa = safe_div(d["prior_net_income"],  d["prior_total_assets"])
    if d["net_income"] and d["net_income"] > 0:                          s += 1
    if d["operating_cash_flow"] and d["operating_cash_flow"] > 0:       s += 1
    if roa is not None and proa is not None and roa > proa:              s += 1
    if (d["operating_cash_flow"] is not None and d["net_income"] is not None
            and d["operating_cash_flow"] > d["net_income"]):             s += 1
    lev  = safe_div(d["total_debt"],        d["total_assets"])
    plev = safe_div(d["prior_total_debt"],   d["prior_total_assets"])
    if lev is not None and plev is not None and lev < plev:              s += 1
    liq  = safe_div(d["current_assets"],    d["current_liabilities"])
    pliq = safe_div(d["prior_current_assets"], d["prior_current_liabilities"])
    if liq is not None and pliq is not None and liq > pliq:              s += 1
    if (d["shares_outstanding"] is not None
            and d["prior_shares_outstanding"] is not None
            and d["shares_outstanding"] <= d["prior_shares_outstanding"]): s += 1
    gm   = safe_div(d["gross_profit"],      d["revenue"])
    pgm  = safe_div(d["prior_gross_profit"], d["prior_revenue"])
    if gm is not None and pgm is not None and gm > pgm:                  s += 1
    at   = safe_div(d["revenue"],           d["total_assets"])
    pat  = safe_div(d["prior_revenue"],     d["prior_total_assets"])
    if at is not None and pat is not None and at > pat:                   s += 1
    return s

def calc_altman_z(d):
    wc  = (d["current_assets"] - d["current_liabilities"]
           if d["current_assets"] is not None and d["current_liabilities"] is not None else None)
    mve = (d["price"] * d["shares_outstanding"]
           if d["price"] is not None and d["shares_outstanding"] is not None else None)
    x1 = safe_div(wc,                    d["total_assets"])
    x2 = safe_div(d["retained_earnings"], d["total_assets"])
    x3 = safe_div(d["ebit"],             d["total_assets"])
    x4 = safe_div(mve,                   d["total_liabilities"])
    x5 = safe_div(d["revenue"],          d["total_assets"])
    if None in (x1, x2, x3, x4, x5): return None
    return 1.2*x1 + 1.4*x2 + 3.3*x3 + 0.6*x4 + x5

def rate_fundamental(roe, de, pe, fs, zs):
    s = 0
    if roe is not None:
        s += 2 if roe >= 0.20 else (1 if roe >= 0.10 else 0)
    if de is not None:
        s += 2 if de <= 0.50 else (1 if de <= 1.50 else (-1 if de > 2.50 else 0))
    if pe is not None:
        s += 2 if pe <= 15 else (1 if pe <= 30 else (-1 if pe > 50 else 0))
    s += 2 if fs >= 8 else (1 if fs >= 6 else (-1 if fs <= 3 else 0))
    if zs is not None:
        s += 2 if zs >= 3 else (1 if zs >= 1.8 else -2)
    rating = "High Buy" if s >= 7 else ("Buy" if s >= 4 else ("Hold" if s >= 1 else "Sell"))
    return s, rating

def fetch_fundamental_metrics(ticker):
    stock = yf.Ticker(ticker)
    info  = stock.get_info()
    if not is_us_stock(info):
        raise ValueError("not a U.S.-listed common stock")
    bal, inc, cf = stock.balance_sheet, stock.income_stmt, stock.cashflow
    price  = info.get("currentPrice") or info.get("regularMarketPrice")
    shares = info.get("sharesOutstanding")
    d = {
        "price":                     float(price)  if price  else None,
        "shares_outstanding":        float(shares) if shares else None,
        "total_assets":              first_available(bal, ["Total Assets"]),
        "total_liabilities":         first_available(bal, ["Total Liabilities Net Minority Interest","Total Liabilities"]),
        "current_assets":            first_available(bal, ["Current Assets","Total Current Assets"]),
        "current_liabilities":       first_available(bal, ["Current Liabilities","Total Current Liabilities"]),
        "retained_earnings":         first_available(bal, ["Retained Earnings"]),
        "total_debt":                first_available(bal, ["Total Debt"]),
        "total_equity":              first_available(bal, ["Stockholders Equity","Total Equity Gross Minority Interest"]),
        "revenue":                   first_available(inc, ["Total Revenue"]),
        "gross_profit":              first_available(inc, ["Gross Profit"]),
        "ebit":                      first_available(inc, ["EBIT","Operating Income"]),
        "net_income":                first_available(inc, ["Net Income","Net Income Common Stockholders"]),
        "operating_cash_flow":       first_available(cf,  ["Operating Cash Flow","Total Cash From Operating Activities"]),
        "prior_net_income":          prior_available(inc, ["Net Income","Net Income Common Stockholders"]),
        "prior_total_assets":        prior_available(bal, ["Total Assets"]),
        "prior_current_assets":      prior_available(bal, ["Current Assets","Total Current Assets"]),
        "prior_current_liabilities": prior_available(bal, ["Current Liabilities","Total Current Liabilities"]),
        "prior_total_debt":          prior_available(bal, ["Total Debt"]),
        "prior_shares_outstanding":  None,
        "prior_gross_profit":        prior_available(inc, ["Gross Profit"]),
        "prior_revenue":             prior_available(inc, ["Total Revenue"]),
    }
    roe    = safe_div(d["net_income"], d["total_equity"])
    de     = safe_div(d["total_debt"], d["total_equity"])
    mktcap = (d["price"] * d["shares_outstanding"]
               if d["price"] and d["shares_outstanding"] else None)
    pe     = safe_div(mktcap, d["net_income"])
    fs     = calc_piotroski(d)
    zs     = calc_altman_z(d)
    fscore, frating = rate_fundamental(roe, de, pe, fs, zs)
    passed = (
        roe is not None and roe >= MIN_ROE  and
        de  is not None and de  <= MAX_DE   and
        pe  is not None and pe  <= MAX_PE   and
        fs  >= MIN_F_SCORE and
        zs  is not None and zs  >= MIN_Z_SCORE
    )
    return FundamentalMetrics(
        ticker=ticker.upper(), roe=roe, debt_to_equity=de, pe_ratio=pe,
        piotroski_f_score=fs, altman_z_score=zs,
        fundamental_score=fscore, fundamental_rating=frating, passed=passed
    )

print("Fundamental helpers ready.")

Fundamental helpers ready.


---
## Cell 5 — Technical screen helpers (no edits needed)
---

In [40]:
def calc_rsi(close, period=14):
    delta  = close.diff()
    gains  = delta.where(delta > 0, 0.0)
    losses = -delta.where(delta < 0, 0.0)
    avg_g  = gains.rolling(period).mean()
    avg_l  = losses.rolling(period).mean()
    rs     = avg_g / avg_l
    rsi    = 100 - (100 / (1 + rs))
    latest = rsi.dropna()
    return float(latest.iloc[-1]) if len(latest) else None

def rate_technical(price, ma50, ma200, rsi14, high52):
    s = 0
    if price and ma50  and price > ma50:   s += 1
    if price and ma200 and price > ma200:  s += 1
    if ma50  and ma200 and ma50  > ma200:  s += 1
    if rsi14 is not None:
        if 40 <= rsi14 <= 70: s += 1
        elif rsi14 > 75:      s -= 1
    dist = None
    if price and high52 and high52 != 0:
        dist = (price - high52) / high52
        if dist >= -0.20: s += 1
    rating = "Bullish" if s >= 4 else ("Neutral" if s >= 2 else "Bearish")
    return s, rating, dist

def fetch_technical_metrics(ticker):
    stock   = yf.Ticker(ticker)
    history = stock.history(period="1y", interval="1d", auto_adjust=True)
    if history.empty: raise ValueError("no price history")
    close   = history["Close"].dropna()
    if len(close) < 200: raise ValueError("< 200 days of history")
    price   = float(close.iloc[-1])
    ma50    = float(close.rolling(50).mean().iloc[-1])
    ma200   = float(close.rolling(200).mean().iloc[-1])
    rsi14   = calc_rsi(close)
    high52  = float(close.max())
    ts, tr, dist = rate_technical(price, ma50, ma200, rsi14, high52)
    return TechnicalMetrics(
        ticker=ticker.upper(), price=price, ma_50=ma50, ma_200=ma200,
        rsi_14=rsi14, high_52w=high52, distance_from_52w_high=dist,
        technical_score=ts, technical_rating=tr
    )

def final_view(f_rating, t_rating):
    if f_rating in ("High Buy","Buy") and t_rating == "Bullish":  return "Strong Candidate"
    if f_rating in ("High Buy","Buy") and t_rating == "Neutral":  return "Watchlist"
    if f_rating in ("High Buy","Buy") and t_rating == "Bearish":  return "Good Company, Wait"
    if f_rating == "Hold"             and t_rating == "Bullish":  return "Trading Candidate"
    return "Avoid"

print("Technical helpers ready.")

Technical helpers ready.


---
## Cell 6 — Run fundamental + technical screens (no edits needed)
---

In [53]:
print(f"Screening {len(TICKERS)} tickers ...\n")

fund_rows, tech_rows = [], []

for t in TICKERS:
    try:
        fund_rows.append(fetch_fundamental_metrics(t))
        print(f"  {t:<6} fundamental OK")
    except Exception as e:
        print(f"  {t:<6} fundamental SKIP ({e})")

for f in fund_rows:
    try:
        tech_rows.append(fetch_technical_metrics(f.ticker))
        print(f"  {f.ticker:<6} technical  OK")
    except Exception as e:
        print(f"  {f.ticker:<6} technical  SKIP ({e})")

tech_map = {r.ticker: r for r in tech_rows}
rows     = [(f, tech_map.get(f.ticker)) for f in fund_rows]
rows.sort(key=lambda x: (
    x[0].fundamental_score,
    x[1].technical_score if x[1] else -999
), reverse=True)

def fmt(v, kind="pct"):
    if v is None: return "n/a"
    if kind == "pct":   return f"{v:.1%}"
    if kind == "num":   return f"{v:.2f}"
    if kind == "money": return f"${v:.2f}"
    return str(v)

screen_data = []
for f, t in rows:
    tr   = t.technical_rating if t else "n/a"
    view = final_view(f.fundamental_rating, tr) if t else "No Tech Data"
    screen_data.append({
        "Ticker"  : f.ticker,
        "ROE"     : fmt(f.roe,             "pct"),
        "D/E"     : fmt(f.debt_to_equity,  "num"),
        "P/E"     : fmt(f.pe_ratio,        "num"),
        "F-Score" : f.piotroski_f_score,
        "Z-Score" : fmt(f.altman_z_score,  "num"),
        "Fund"    : f.fundamental_rating,
        "Price"   : fmt(t.price  if t else None, "money"),
        "50DMA"   : fmt(t.ma_50  if t else None, "money"),
        "200DMA"  : fmt(t.ma_200 if t else None, "money"),
        "RSI"     : fmt(t.rsi_14 if t else None, "num"),
        "Tech"    : tr,
        "View"    : view,
        "Passed"  : "✓" if f.passed else "",
    })

screen_df = pd.DataFrame(screen_data)

def highlight_view(val):
    colors = {
        "Strong Candidate" : "background-color:#d4edda;color:#155724",
        "Watchlist"        : "background-color:#d1ecf1;color:#0c5460",
        "Trading Candidate": "background-color:#fff3cd;color:#856404",
        "Good Company, Wait":"background-color:#ffeeba;color:#856404",
        "Avoid"            : "background-color:#f8d7da;color:#721c24",
    }
    return colors.get(val, "")

display(
    screen_df.style
        .map(highlight_view, subset=["View"])
        .set_caption(f"Fundamental + Technical Screen — {ANALYSIS_DATE}")
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size","14px"),("font-weight","bold"),("text-align","left")]
        }])
)

candidates     = [f.ticker for f, t in rows if t and final_view(f.fundamental_rating, t.technical_rating) == "Strong Candidate"]
watchlist      = [f.ticker for f, t in rows if t and final_view(f.fundamental_rating, t.technical_rating) == "Watchlist"]
passed_tickers = [f.ticker for f, _ in rows if f.passed]

print(f"\nStrong Candidates : {candidates}")
print(f"Watchlist         : {watchlist}")
print(f"All passed screen : {passed_tickers}")

Screening 8 tickers ...

  AAPL   fundamental OK
  MSFT   fundamental OK
  NVDA   fundamental OK
  AMZN   fundamental OK
  GOOGL  fundamental OK
  META   fundamental OK
  JPM    fundamental OK
  TSLA   fundamental OK
  AAPL   technical  OK
  MSFT   technical  OK
  NVDA   technical  OK
  AMZN   technical  OK
  GOOGL  technical  OK
  META   technical  OK
  JPM    technical  OK
  TSLA   technical  OK


,Ticker,ROE,D/E,P/E,F-Score,Z-Score,Fund,Price,50DMA,200DMA,RSI,Tech,View,Passed
0,META,27.8%,0.39,20.42,4,7.00,High Buy,$562.20,$618.75,$651.93,40.39,Bearish,"Good Company, Wait",
1,MSFT,29.6%,0.18,27.28,5,7.81,High Buy,$373.94,$412.42,$448.23,16.91,Bearish,"Good Company, Wait",
2,AAPL,151.9%,1.34,38.59,7,11.35,Buy,$294.30,$290.05,$268.37,30.41,Bullish,Strong Candidate,
3,AMZN,18.9%,0.37,32.42,6,5.44,Buy,$234.11,$257.00,$232.83,33.60,Neutral,Watchlist,
4,GOOGL,31.8%,0.14,15.37,5,n/a,Buy,$346.13,$368.40,$311.90,40.99,Bullish,Strong Candidate,
5,NVDA,76.3%,0.07,40.35,3,63.56,Buy,$200.04,$209.85,$190.02,34.80,Neutral,Watchlist,
6,TSLA,4.6%,0.18,377.76,6,17.19,Buy,$381.61,$404.33,$417.53,36.36,Bearish,"Good Company, Wait",
7,JPM,15.7%,1.38,15.69,2,n/a,Hold,$334.14,$309.77,$305.54,76.69,Neutral,Avoid,



Strong Candidates : ['AAPL', 'GOOGL']
Watchlist         : ['AMZN', 'NVDA']
All passed screen : []


---
## Cell 7 — 📰 News Intelligence Layer (RSS + yfinance company news)
**No API key required.** Uses `feedparser` for market RSS feeds and `yfinance` for per-ticker company news.

---

In [42]:
import feedparser

# ─── RSS Feed sources (free, no key needed) ────────────────────────────────
NEWS_SOURCES = {
    "Reuters"  : "https://feeds.reuters.com/reuters/businessNews",
    "Yahoo"    : "https://finance.yahoo.com/news/rssindex",
    "FT"       : "https://www.ft.com/rss/home",
    "WSJ"      : "https://feeds.a.dj.com/rss/RSSMarketsMain.xml",
}

# ─── Sentiment word lists ─────────────────────────────────────────────────
BULLISH_WORDS = [
    "growth", "beat", "upgrade", "strong", "record", "surge", "rally",
    "profit", "expansion", "bullish", "outperform", "raise guidance",
    "dividend", "buyback", "acquisition", "innovation", "breakthrough",
]
BEARISH_WORDS = [
    "miss", "downgrade", "risk", "recession", "decline", "warning",
    "layoff", "bankruptcy", "fraud", "investigation", "tariff",
    "inflation", "slowdown", "debt", "loss", "cut guidance",
]

def get_market_news(max_per_source: int = 15) -> list[dict]:
    """Collect headlines from all RSS sources."""
    articles = []
    for source, url in NEWS_SOURCES.items():
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries[:max_per_source]:
                articles.append({
                    "source": source,
                    "title" : getattr(entry, "title", ""),
                    "link"  : getattr(entry, "link",  ""),
                })
            print(f"  {source:<10} {len(feed.entries[:max_per_source])} articles fetched")
        except Exception as e:
            print(f"  {source:<10} SKIP ({e})")
    return articles

def score_market_news(articles: list[dict]) -> tuple[int, str]:
    """Keyword sentiment score across all market news headlines."""
    score = 0
    for article in articles:
        title = article["title"].lower()
        for w in BULLISH_WORDS:
            if w in title: score += 1
        for w in BEARISH_WORDS:
            if w in title: score -= 1
    if score >= 5:    sentiment = "Bullish"
    elif score >= 1:  sentiment = "Mildly Bullish"
    elif score == 0:  sentiment = "Neutral"
    elif score >= -4: sentiment = "Mildly Bearish"
    else:             sentiment = "Bearish"
    return score, sentiment

def get_company_news(ticker: str, max_items: int = 10) -> list[str]:
    """Per-ticker news from yfinance (already in your stack, no API key)."""
    try:
        stock = yf.Ticker(ticker)
        news  = stock.news or []
        return [item.get("content", {}).get("title", "") or item.get("title", "")
                for item in news[:max_items]
                if item.get("content", {}).get("title") or item.get("title")]
    except Exception:
        return []

def score_company_news(headlines: list[str]) -> tuple[int, float]:
    """Return raw score and normalized 0–10 score for a single company."""
    raw = 0
    for title in headlines:
        t = title.lower()
        for w in BULLISH_WORDS:
            if w in t: raw += 1
        for w in BEARISH_WORDS:
            if w in t: raw -= 1
    # Normalize: clip to [-5, +5] then map to [0, 10]
    clipped    = max(-5, min(5, raw))
    normalized = (clipped + 5)           # 0..10
    return raw, float(normalized)

# ─── Run news collection ──────────────────────────────────────────────────
print("Collecting market-wide RSS news ...")
market_articles  = get_market_news()
market_raw_score, market_sentiment = score_market_news(market_articles)

print(f"\nMarket Sentiment Score : {market_raw_score:+d}  →  {market_sentiment}")
print(f"Total articles scanned : {len(market_articles)}")

# ─── Per-ticker company news ──────────────────────────────────────────────
print("\nCollecting per-ticker company news ...")
company_news_map   = {}   # ticker → [headlines]
company_news_scores = {}  # ticker → (raw, normalized 0–10)

for f, _ in rows:
    headlines = get_company_news(f.ticker)
    company_news_map[f.ticker]    = headlines
    company_news_scores[f.ticker] = score_company_news(headlines)
    raw, norm = company_news_scores[f.ticker]
    print(f"  {f.ticker:<6}  {len(headlines)} headlines  raw={raw:+d}  news_score={norm:.1f}/10")

print("\nNews layer complete.")

  Reuters    0 articles fetched
  Yahoo      15 articles fetched
  FT         11 articles fetched
  WSJ        15 articles fetched

Market Sentiment Score : +3  →  Mildly Bullish
Total articles scanned : 41

  META    10 headlines  raw=+1  news_score=6.0/10
  MSFT    10 headlines  raw=+3  news_score=8.0/10
  AAPL    10 headlines  raw=+1  news_score=6.0/10
  AMZN    10 headlines  raw=+1  news_score=6.0/10
  GOOGL   10 headlines  raw=+1  news_score=6.0/10
  NVDA    10 headlines  raw=+2  news_score=7.0/10
  TSLA    10 headlines  raw=+0  news_score=5.0/10
  JPM     10 headlines  raw=+0  news_score=5.0/10

News layer complete.


---
## Cell 8 — 🏛️ FRED Macro Score (requires your FRED API key)
Pulls 6 economic indicators from the Federal Reserve and converts them into a single 0–10 macro score.

| Series | Indicator |
|--------|-----------|
| FEDFUNDS | Fed Funds Rate |
| CPIAUCSL | CPI Inflation |
| UNRATE | Unemployment Rate |
| T10Y2Y | 10Y-2Y Yield Spread (recession signal) |
| VIXCLS | VIX Fear Index |
| A191RL1Q225SBEA | Real GDP Growth |

---

In [43]:
from fredapi import Fred

@dataclass
class MacroEnvironment:
    fed_funds_rate:   float | None
    cpi_yoy:          float | None
    unemployment:     float | None
    yield_spread:     float | None
    vix:              float | None
    gdp_growth:       float | None
    macro_score:      float          # 0–10
    macro_label:      str
    macro_summary:    str

def _latest_fred(fred: Fred, series_id: str) -> float | None:
    """Pull the most recent non-null value for a FRED series."""
    try:
        s = fred.get_series(series_id)
        s = s.dropna()
        return float(s.iloc[-1]) if len(s) else None
    except Exception:
        return None

def _cpi_yoy(fred: Fred) -> float | None:
    """Calculate CPI year-over-year % change."""
    try:
        s = fred.get_series("CPIAUCSL").dropna()
        if len(s) < 13: return None
        latest_val = float(s.iloc[-1])
        prior_val  = float(s.iloc[-13])
        return ((latest_val - prior_val) / prior_val) * 100
    except Exception:
        return None

def fetch_macro_environment(api_key: str) -> MacroEnvironment:
    fred = Fred(api_key=api_key)

    fed   = _latest_fred(fred, "FEDFUNDS")
    cpi   = _cpi_yoy(fred)
    unemp = _latest_fred(fred, "UNRATE")
    t10y2 = _latest_fred(fred, "T10Y2Y")
    vix   = _latest_fred(fred, "VIXCLS")
    gdp   = _latest_fred(fred, "A191RL1Q225SBEA")

    # ── Scoring rubric (each indicator contributes 0, 1, or 2 points) ────────
    score = 0.0

    # Fed Funds Rate: lower = more accommodative
    if fed is not None:
        if fed <= 2.0:   score += 2   # Very accommodative
        elif fed <= 4.0: score += 1   # Neutral-ish
        else:            score += 0   # Restrictive

    # CPI inflation: lower = better for equities
    if cpi is not None:
        if cpi <= 2.5:   score += 2   # On target
        elif cpi <= 4.0: score += 1   # Elevated but manageable
        else:            score += 0   # Hot / stagflation risk

    # Unemployment: lower = stronger economy
    if unemp is not None:
        if unemp <= 4.0:   score += 2   # Full employment
        elif unemp <= 5.5: score += 1   # Softening
        else:              score += 0   # Weakening labor market

    # Yield spread (10Y-2Y): positive = normal curve, negative = inversion
    if t10y2 is not None:
        if t10y2 >= 0.50:    score += 2   # Healthy curve
        elif t10y2 >= 0.0:   score += 1   # Flat but no inversion
        else:                score += 0   # Inverted (recession signal)

    # VIX: lower = calmer markets
    if vix is not None:
        if vix <= 15:   score += 2   # Low fear
        elif vix <= 25: score += 1   # Moderate
        else:           score += 0   # High fear / volatility

    # GDP growth: positive and robust = good
    if gdp is not None:
        if gdp >= 3.0:   score += 2   # Strong growth
        elif gdp >= 1.0: score += 1   # Moderate
        else:            score += 0   # Stall/contraction risk

    # Max raw score = 12 → normalize to 0–10
    max_raw     = 12.0
    norm_score  = (score / max_raw) * 10

    if norm_score >= 7.5:   label = "Macro Tailwind"
    elif norm_score >= 5.0: label = "Macro Neutral"
    elif norm_score >= 2.5: label = "Macro Headwind"
    else:                   label = "Macro Risk-Off"

    # Build a human-readable summary
    parts = []
    if fed   is not None: parts.append(f"Fed Funds {fed:.2f}%")
    if cpi   is not None: parts.append(f"CPI YoY {cpi:.1f}%")
    if unemp is not None: parts.append(f"Unemployment {unemp:.1f}%")
    if t10y2 is not None: parts.append(f"10Y-2Y Spread {t10y2:.2f}%")
    if vix   is not None: parts.append(f"VIX {vix:.1f}")
    if gdp   is not None: parts.append(f"GDP Growth {gdp:.1f}%")
    summary = " | ".join(parts) if parts else "No data"

    return MacroEnvironment(
        fed_funds_rate=fed, cpi_yoy=cpi, unemployment=unemp,
        yield_spread=t10y2, vix=vix, gdp_growth=gdp,
        macro_score=norm_score, macro_label=label, macro_summary=summary
    )

# ─── Run macro fetch ─────────────────────────────────────────────────────────
print("Fetching FRED macro indicators ...")

if FRED_API_KEY:
    macro = fetch_macro_environment(FRED_API_KEY)
    print(f"\n{'─'*55}")
    print(f"  Fed Funds Rate     : {macro.fed_funds_rate:.2f}%"  if macro.fed_funds_rate is not None else "  Fed Funds Rate     : n/a")
    print(f"  CPI (YoY)          : {macro.cpi_yoy:.1f}%"         if macro.cpi_yoy       is not None else "  CPI (YoY)          : n/a")
    print(f"  Unemployment       : {macro.unemployment:.1f}%"    if macro.unemployment   is not None else "  Unemployment       : n/a")
    print(f"  10Y-2Y Spread      : {macro.yield_spread:.2f}%"    if macro.yield_spread   is not None else "  10Y-2Y Spread      : n/a")
    print(f"  VIX                : {macro.vix:.1f}"              if macro.vix            is not None else "  VIX                : n/a")
    print(f"  Real GDP Growth    : {macro.gdp_growth:.1f}%"      if macro.gdp_growth     is not None else "  Real GDP Growth    : n/a")
    print(f"{'─'*55}")
    print(f"  Macro Score        : {macro.macro_score:.1f} / 10")
    print(f"  Macro Environment  : {macro.macro_label}")
    print(f"{'─'*55}")
else:
    # Fallback: neutral macro if no key
    macro = MacroEnvironment(
        fed_funds_rate=None, cpi_yoy=None, unemployment=None,
        yield_spread=None, vix=None, gdp_growth=None,
        macro_score=5.0, macro_label="Macro Neutral (no key)",
        macro_summary="FRED_API_KEY not set — macro defaulted to 5.0/10"
    )
    print("  FRED_API_KEY not set. Macro score defaulted to 5.0/10.")

print("\nMacro layer complete.")

Fetching FRED macro indicators ...

───────────────────────────────────────────────────────
  Fed Funds Rate     : 3.63%
  CPI (YoY)          : 4.3%
  Unemployment       : 4.3%
  10Y-2Y Spread      : 0.27%
  VIX                : 17.3
  Real GDP Growth    : 1.6%
───────────────────────────────────────────────────────
  Macro Score        : 4.2 / 10
  Macro Environment  : Macro Headwind
───────────────────────────────────────────────────────

Macro layer complete.


---
## Cell 9 — 📊 Composite Score (4-factor weighted model)
Combines Fundamental + Technical + News + Macro into a single 0–10 score per ticker.

**Weights:** 40% Fundamental · 25% Technical · 20% News · 15% Macro

---

In [54]:
@dataclass
class CompositeScore:
    ticker:          str
    fund_score_10:   float   # fundamental score normalized 0–10
    tech_score_10:   float   # technical score normalized 0–10
    news_score_10:   float   # company news score 0–10
    macro_score_10:  float   # macro score 0–10
    composite:       float   # weighted composite 0–10
    recommendation:  str
    action:          str

def normalize_fundamental(raw: int) -> float:
    """Map fundamental score (-4 to +10) → 0–10."""
    # Observed range: -4 (all negatives) to +10 (all positives)
    return max(0.0, min(10.0, (raw + 4) / 14 * 10))

def normalize_technical(raw: int) -> float:
    """Map technical score (0–5) → 0–10."""
    return max(0.0, min(10.0, raw / 5 * 10))

def composite_recommendation(score: float) -> tuple[str, str]:
    if score >= 7.5:   return "Strong Buy",  "🟢 STRONG BUY"
    if score >= 6.0:   return "Buy",         "🟢 BUY"
    if score >= 5.0:   return "Watch",       "🔵 WATCH"
    if score >= 3.5:   return "Hold",        "🟡 HOLD"
    return                     "Avoid",       "🔴 AVOID"

# ─── Build composite scores ───────────────────────────────────────────────
composite_scores: dict[str, CompositeScore] = {}

for f, t in rows:
    f10  = normalize_fundamental(f.fundamental_score)
    t10  = normalize_technical(t.technical_score) if t else 5.0   # neutral if missing
    n10  = company_news_scores.get(f.ticker, (0, 5.0))[1]         # per-ticker news score
    m10  = macro.macro_score                                        # same for all tickers

    comp = (W_FUNDAMENTAL * f10 +
            W_TECHNICAL   * t10 +
            W_NEWS        * n10 +
            W_MACRO       * m10)

    rec, action = composite_recommendation(comp)

    composite_scores[f.ticker] = CompositeScore(
        ticker=f.ticker,
        fund_score_10=f10, tech_score_10=t10,
        news_score_10=n10, macro_score_10=m10,
        composite=comp, recommendation=rec, action=action
    )

# ─── Display composite table ──────────────────────────────────────────────
comp_data = []
for f, t in rows:
    cs = composite_scores[f.ticker]
    tr = t.technical_rating if t else "n/a"
    comp_data.append({
        "Ticker"   : f.ticker,
        "Fund"     : f"{cs.fund_score_10:.1f}",
        "Tech"     : f"{cs.tech_score_10:.1f}",
        "News"     : f"{cs.news_score_10:.1f}",
        "Macro"    : f"{cs.macro_score_10:.1f}",
        "Composite": f"{cs.composite:.1f}",
        "Action"   : cs.action,
        "Passed ✓" : "✓" if f.passed else "",
    })

comp_df = pd.DataFrame(comp_data)

def highlight_action(val):
    colors = {
        "🟢 STRONG BUY": "background-color:#155724;color:#d4edda;font-weight:bold",
        "🟢 BUY"       : "background-color:#d4edda;color:#155724",
        "🔵 WATCH"     : "background-color:#d1ecf1;color:#0c5460",
        "🟡 HOLD"      : "background-color:#fff3cd;color:#856404",
        "🔴 AVOID"     : "background-color:#f8d7da;color:#721c24",
    }
    return colors.get(val, "")

display(
    comp_df.style
        .map(highlight_action, subset=["Action"])
        .set_caption(
            f"4-Factor Composite Score — {ANALYSIS_DATE}  "
            f"| Market Sentiment: {market_sentiment} ({market_raw_score:+d}) "
            f"| Macro: {macro.macro_label} ({macro.macro_score:.1f}/10)"
        )
        .set_table_styles([{
            "selector": "caption",
            "props": [("font-size","13px"),("font-weight","bold"),("text-align","left")]
        }])
)

# Summary
top_picks = [t for t, cs in composite_scores.items() if cs.recommendation in ("Strong Buy","Buy")]
print(f"\nTop picks (composite score ≥ 6.0) : {top_picks}")
print(f"Market Sentiment : {market_sentiment} ({market_raw_score:+d})")
print(f"Macro Environment: {macro.macro_label} ({macro.macro_score:.1f}/10)")

,Ticker,Fund,Tech,News,Macro,Composite,Action,Passed ✓
0,META,7.9,2.0,6.0,4.2,5.5,🔵 WATCH,
1,MSFT,7.9,0.0,8.0,4.2,5.4,🔵 WATCH,
2,AAPL,7.1,8.0,6.0,4.2,6.7,🟢 BUY,
3,AMZN,7.1,6.0,6.0,4.2,6.2,🟢 BUY,
4,GOOGL,6.4,8.0,6.0,4.2,6.4,🟢 BUY,
5,NVDA,6.4,6.0,7.0,4.2,6.1,🟢 BUY,
6,TSLA,5.7,0.0,5.0,4.2,3.9,🟡 HOLD,
7,JPM,4.3,6.0,5.0,4.2,4.8,🟡 HOLD,



Top picks (composite score ≥ 6.0) : ['AAPL', 'AMZN', 'GOOGL', 'NVDA']
Market Sentiment : Mildly Bullish (+3)
Macro Environment: Macro Headwind (4.2/10)


---
## Cell 10 — 🧠 Upgraded Decision Engine (4-factor report)
Generates a full decision report per ticker including Fundamental, Technical, News, Macro, and Final Composite decision.

---

In [45]:
from IPython.display import Markdown, display as ipy_display

def rule_fundamental_report(f: FundamentalMetrics) -> str:
    lines = ["### 📊 Fundamental Analysis"]
    if f.roe is not None:
        verdict = 'Strong (≥20%)' if f.roe >= 0.20 else ('Adequate (10–20%)' if f.roe >= 0.10 else 'Weak (<10%)')
        lines.append(f'- **Return on Equity (ROE):** {f.roe:.1%} — {verdict}')
    else:
        lines.append('- **Return on Equity (ROE):** n/a')
    if f.debt_to_equity is not None:
        verdict = 'Low leverage (≤0.5)' if f.debt_to_equity <= 0.5 else ('Moderate (0.5–1.5)' if f.debt_to_equity <= 1.5 else ('Elevated (1.5–2.5)' if f.debt_to_equity <= 2.5 else 'High risk (>2.5)'))
        lines.append(f'- **Debt-to-Equity (D/E):** {f.debt_to_equity:.2f} — {verdict}')
    else:
        lines.append('- **Debt-to-Equity (D/E):** n/a')
    if f.pe_ratio is not None:
        verdict = 'Undervalued (≤15)' if f.pe_ratio <= 15 else ('Fair value (15–30)' if f.pe_ratio <= 30 else ('Stretched (30–50)' if f.pe_ratio <= 50 else 'Expensive (>50)'))
        lines.append(f'- **Price-to-Earnings (P/E):** {f.pe_ratio:.1f}x — {verdict}')
    else:
        lines.append('- **Price-to-Earnings (P/E):** n/a')
    fs_verdict = 'Excellent (8–9)' if f.piotroski_f_score >= 8 else ('Good (6–7)' if f.piotroski_f_score >= 6 else ('Mediocre (4–5)' if f.piotroski_f_score >= 4 else 'Poor (≤3)'))
    lines.append(f'- **Piotroski F-Score:** {f.piotroski_f_score}/9 — {fs_verdict}')
    if f.altman_z_score is not None:
        zv = 'Safe zone (≥3.0)' if f.altman_z_score >= 3.0 else ('Grey zone (1.8–3.0)' if f.altman_z_score >= 1.8 else 'Distress zone (<1.8)')
        lines.append(f'- **Altman Z-Score:** {f.altman_z_score:.2f} — {zv}')
    else:
        lines.append('- **Altman Z-Score:** n/a')
    lines.append(f'\n**Overall Fundamental Rating: {f.fundamental_rating}**')
    return '\n'.join(lines)


def rule_technical_report(t: TechnicalMetrics) -> str:
    lines = ["### 📈 Technical Analysis"]
    if t.price is not None:
        lines.append(f'- **Current Price:** ${t.price:.2f}')
    if t.price and t.ma_50:
        rel = 'above' if t.price > t.ma_50 else 'below'
        lines.append(f'- **50-Day MA:** ${t.ma_50:.2f} — price is {rel} (momentum {"positive" if rel=="above" else "negative"})')
    if t.price and t.ma_200:
        rel = 'above' if t.price > t.ma_200 else 'below'
        lines.append(f'- **200-Day MA:** ${t.ma_200:.2f} — price is {rel} (long-term trend {"up" if rel=="above" else "down"})')
    if t.ma_50 and t.ma_200:
        cross = 'Golden Cross (bullish)' if t.ma_50 > t.ma_200 else 'Death Cross (bearish)'
        lines.append(f'- **MA Cross:** {cross}')
    if t.rsi_14 is not None:
        rsi_v = 'Overbought — caution' if t.rsi_14 > 75 else ('Healthy range' if t.rsi_14 >= 40 else 'Oversold — potential reversal')
        lines.append(f'- **RSI (14):** {t.rsi_14:.1f} — {rsi_v}')
    if t.distance_from_52w_high is not None:
        pct = t.distance_from_52w_high * 100
        strength = 'Near 52-week high (strong momentum)' if pct >= -5 else ('Within 20% of high' if pct >= -20 else 'More than 20% off 52-week high')
        lines.append(f'- **Distance from 52W High:** {pct:.1f}% — {strength}')
    lines.append(f'\n**Overall Technical Rating: {t.technical_rating}** (score {t.technical_score}/5)')
    return '\n'.join(lines)


def rule_news_report(ticker: str, news_score_10: float, market_sent: str) -> str:
    lines = ["### 📰 News Intelligence"]
    headlines = company_news_map.get(ticker, [])
    raw, norm = company_news_scores.get(ticker, (0, 5.0))

    lines.append(f'- **Company News Score:** {norm:.1f}/10 (raw keyword score: {raw:+d})')
    lines.append(f'- **Market-Wide Sentiment:** {market_sent} ({market_raw_score:+d})')

    if headlines:
        lines.append(f'\n**Recent Headlines ({len(headlines)}):**')
        for h in headlines[:5]:
            lines.append(f'  - {h}')
        if len(headlines) > 5:
            lines.append(f'  - _(+ {len(headlines)-5} more)_')
    else:
        lines.append('\n_No company-specific headlines found via yfinance._')

    return '\n'.join(lines)


def rule_macro_report(m: MacroEnvironment) -> str:
    lines = ["### 🏛️ Macro Environment (FRED)"]
    if m.fed_funds_rate is not None:
        lvl = 'Restrictive (>4%)' if m.fed_funds_rate > 4 else ('Neutral (2–4%)' if m.fed_funds_rate >= 2 else 'Accommodative (<2%)')
        lines.append(f'- **Fed Funds Rate:** {m.fed_funds_rate:.2f}% — {lvl}')
    if m.cpi_yoy is not None:
        lvl = 'On target (≤2.5%)' if m.cpi_yoy <= 2.5 else ('Elevated (2.5–4%)' if m.cpi_yoy <= 4 else 'Hot inflation (>4%)')
        lines.append(f'- **CPI Inflation (YoY):** {m.cpi_yoy:.1f}% — {lvl}')
    if m.unemployment is not None:
        lvl = 'Full employment (≤4%)' if m.unemployment <= 4 else ('Softening (4–5.5%)' if m.unemployment <= 5.5 else 'Weakening (>5.5%)')
        lines.append(f'- **Unemployment:** {m.unemployment:.1f}% — {lvl}')
    if m.yield_spread is not None:
        lvl = 'Healthy curve (>0.5%)' if m.yield_spread >= 0.5 else ('Flat (0–0.5%)' if m.yield_spread >= 0 else 'Inverted — recession signal')
        lines.append(f'- **10Y-2Y Yield Spread:** {m.yield_spread:.2f}% — {lvl}')
    if m.vix is not None:
        lvl = 'Low fear (≤15)' if m.vix <= 15 else ('Moderate (15–25)' if m.vix <= 25 else 'High fear (>25)')
        lines.append(f'- **VIX:** {m.vix:.1f} — {lvl}')
    if m.gdp_growth is not None:
        lvl = 'Strong (≥3%)' if m.gdp_growth >= 3 else ('Moderate (1–3%)' if m.gdp_growth >= 1 else 'Stall risk (<1%)')
        lines.append(f'- **Real GDP Growth:** {m.gdp_growth:.1f}% — {lvl}')
    lines.append(f'\n**Macro Score: {m.macro_score:.1f}/10 — {m.macro_label}**')
    return '\n'.join(lines)


def rule_risk_report(f: FundamentalMetrics, t: TechnicalMetrics | None) -> str:
    lines = ["### ⚠️ Risk Assessment"]
    risks, positives = [], []
    if f.debt_to_equity is not None and f.debt_to_equity > 2.0:
        risks.append(f'High leverage (D/E {f.debt_to_equity:.2f}) increases bankruptcy risk in downturns')
    if f.altman_z_score is not None and f.altman_z_score < 1.8:
        risks.append(f'Altman Z-Score {f.altman_z_score:.2f} signals financial distress')
    if f.pe_ratio is not None and f.pe_ratio > 50:
        risks.append(f'Elevated P/E of {f.pe_ratio:.1f}x — valuation highly sensitive to growth misses')
    if f.piotroski_f_score <= 3:
        risks.append(f'Low F-Score ({f.piotroski_f_score}/9) indicates deteriorating fundamentals')
    if f.roe is not None and f.roe < 0:
        risks.append('Negative ROE — company is destroying shareholder value')
    if t:
        if t.rsi_14 and t.rsi_14 > 75:
            risks.append(f'RSI {t.rsi_14:.1f} — overbought, short-term pullback likely')
        if t.price and t.ma_200 and t.price < t.ma_200:
            risks.append('Price below 200-day MA — long-term downtrend in place')
        if t.distance_from_52w_high is not None and t.distance_from_52w_high < -0.30:
            risks.append(f'Stock is {abs(t.distance_from_52w_high)*100:.0f}% off its 52-week high')
    if macro.yield_spread is not None and macro.yield_spread < 0:
        risks.append('Yield curve inverted — historical recession signal')
    if macro.vix is not None and macro.vix > 25:
        risks.append(f'VIX {macro.vix:.1f} — elevated market fear/uncertainty')
    if f.roe is not None and f.roe >= 0.20:    positives.append('Strong ROE above 20%')
    if f.altman_z_score is not None and f.altman_z_score >= 3.0: positives.append('Z-Score in safe zone')
    if f.piotroski_f_score >= 8:               positives.append('Excellent F-Score signals improving operations')
    if t and t.technical_rating == 'Bullish':  positives.append('All technical indicators aligned bullishly')
    if f.passed:                               positives.append('Passed all quantitative screen filters')
    if macro.macro_label == "Macro Tailwind":  positives.append('Favourable macro backdrop (FRED data)')
    if risks:
        lines.append('**Key Risks:**')
        for r in risks: lines.append(f'  - {r}')
    else:
        lines.append('**Key Risks:** No major red flags identified.')
    if positives:
        lines.append('\n**Strengths:**')
        for p in positives: lines.append(f'  - {p}')
    return '\n'.join(lines)


def rule_composite_decision(f: FundamentalMetrics, t: TechnicalMetrics | None, cs: CompositeScore) -> str:
    lines = ["### 🏆 Portfolio Manager — Final Decision (4-Factor)"]
    lines.append(f'\n## {cs.action}')
    lines.append(f'**Composite Score: {cs.composite:.1f} / 10**  ({cs.recommendation})')
    lines.append(f'\n| Factor | Score | Weight | Contribution |')
    lines.append(f'|--------|-------|--------|-------------|')
    lines.append(f'| Fundamental | {cs.fund_score_10:.1f}/10 | {W_FUNDAMENTAL:.0%} | {cs.fund_score_10*W_FUNDAMENTAL:.2f} |')
    lines.append(f'| Technical   | {cs.tech_score_10:.1f}/10 | {W_TECHNICAL:.0%}   | {cs.tech_score_10*W_TECHNICAL:.2f} |')
    lines.append(f'| News        | {cs.news_score_10:.1f}/10 | {W_NEWS:.0%}        | {cs.news_score_10*W_NEWS:.2f} |')
    lines.append(f'| Macro       | {cs.macro_score_10:.1f}/10 | {W_MACRO:.0%}      | {cs.macro_score_10*W_MACRO:.2f} |')
    lines.append(f'| **Composite** | **{cs.composite:.1f}/10** | 100% | — |')

    action_guidance = {
        "Strong Buy" : "All four factors aligned positively. Consider entering at current levels or on a minor pullback. Set stop-loss below the 200-day MA.",
        "Buy"        : "Predominantly positive signal. Good entry opportunity with manageable risk. Monitor news flow for any deterioration.",
        "Watch"      : "Above-average score but not fully confirmed. Add to watchlist and enter on improvement in the weakest factor.",
        "Hold"       : "Mixed signals across factors. No clear catalyst in either direction. Reassess next quarter.",
        "Avoid"      : "Multiple factors negative. No position warranted. Re-evaluate when fundamentals or technicals improve.",
    }
    lines.append(f'\n**Guidance:** {action_guidance.get(cs.recommendation, "")}')
    return '\n'.join(lines)


def generate_full_report(f: FundamentalMetrics, t: TechnicalMetrics | None, date: str) -> str:
    cs = composite_scores[f.ticker]
    sections = [
        f'# {f.ticker} — Decision Report ({date})',
        rule_fundamental_report(f),
        rule_technical_report(t) if t else '### 📈 Technical Analysis\n_No technical data available._',
        rule_news_report(f.ticker, cs.news_score_10, market_sentiment),
        rule_macro_report(macro),
        rule_risk_report(f, t),
        rule_composite_decision(f, t, cs),
    ]
    return '\n\n---\n\n'.join(sections)


# ─── Generate reports for all tickers ────────────────────────────────────────
decision_reports = {}
for f, t in rows:
    decision_reports[f.ticker] = generate_full_report(f, t, ANALYSIS_DATE)

print(f"Decision reports generated for: {list(decision_reports.keys())}")

Decision reports generated for: ['META', 'MSFT', 'AAPL', 'AMZN', 'GOOGL', 'NVDA', 'TSLA', 'JPM']


---
## Cell 11 — Display full decision reports (no edits needed)
---

In [46]:
from IPython.display import Markdown, display as ipy_display

for ticker, report in decision_reports.items():
    ipy_display(Markdown(report))
    ipy_display(Markdown('---'))

# META — Decision Report (2026-06-23)

---

### 📊 Fundamental Analysis
- **Return on Equity (ROE):** 27.8% — Strong (≥20%)
- **Debt-to-Equity (D/E):** 0.39 — Low leverage (≤0.5)
- **Price-to-Earnings (P/E):** 20.5x — Fair value (15–30)
- **Piotroski F-Score:** 4/9 — Mediocre (4–5)
- **Altman Z-Score:** 7.01 — Safe zone (≥3.0)

**Overall Fundamental Rating: High Buy**

---

### 📈 Technical Analysis
- **Current Price:** $564.16
- **50-Day MA:** $618.78 — price is below (momentum negative)
- **200-Day MA:** $651.94 — price is below (long-term trend down)
- **MA Cross:** Death Cross (bearish)
- **RSI (14):** 40.9 — Healthy range
- **Distance from 52W High:** -28.4% — More than 20% off 52-week high

**Overall Technical Rating: Bearish** (score 1/5)

---

### 📰 News Intelligence
- **Company News Score:** 6.0/10 (raw keyword score: +1)
- **Market-Wide Sentiment:** Mildly Bullish (+3)

**Recent Headlines (10):**
  - Meta debuts AI-powered Meta Glasses, starting at $299
  - What's Going on with Alphabet Stock Tuesday
  - The AI Trade Is Wobbling. Berkshire Hathaway Was Built for a Market Like This.
  - Meta prediction markets app Arena sinks DraftKings, Robinhood stock
  - AI stock slump raises the question if investors are just taking profits or getting very nervous
  - _(+ 5 more)_

---

### 🏛️ Macro Environment (FRED)
- **Fed Funds Rate:** 3.63% — Neutral (2–4%)
- **CPI Inflation (YoY):** 4.3% — Hot inflation (>4%)
- **Unemployment:** 4.3% — Softening (4–5.5%)
- **10Y-2Y Yield Spread:** 0.27% — Flat (0–0.5%)
- **VIX:** 17.3 — Moderate (15–25)
- **Real GDP Growth:** 1.6% — Moderate (1–3%)

**Macro Score: 4.2/10 — Macro Headwind**

---

### ⚠️ Risk Assessment
**Key Risks:**
  - Price below 200-day MA — long-term downtrend in place

**Strengths:**
  - Strong ROE above 20%
  - Z-Score in safe zone

---

### 🏆 Portfolio Manager — Final Decision (4-Factor)

## 🔵 WATCH
**Composite Score: 5.5 / 10**  (Watch)

| Factor | Score | Weight | Contribution |
|--------|-------|--------|-------------|
| Fundamental | 7.9/10 | 40% | 3.14 |
| Technical   | 2.0/10 | 25%   | 0.50 |
| News        | 6.0/10 | 20%        | 1.20 |
| Macro       | 4.2/10 | 15%      | 0.62 |
| **Composite** | **5.5/10** | 100% | — |

**Guidance:** Above-average score but not fully confirmed. Add to watchlist and enter on improvement in the weakest factor.

---

# MSFT — Decision Report (2026-06-23)

---

### 📊 Fundamental Analysis
- **Return on Equity (ROE):** 29.6% — Strong (≥20%)
- **Debt-to-Equity (D/E):** 0.18 — Low leverage (≤0.5)
- **Price-to-Earnings (P/E):** 27.2x — Fair value (15–30)
- **Piotroski F-Score:** 5/9 — Mediocre (4–5)
- **Altman Z-Score:** 7.80 — Safe zone (≥3.0)

**Overall Fundamental Rating: High Buy**

---

### 📈 Technical Analysis
- **Current Price:** $373.48
- **50-Day MA:** $412.41 — price is below (momentum negative)
- **200-Day MA:** $448.22 — price is below (long-term trend down)
- **MA Cross:** Death Cross (bearish)
- **RSI (14):** 16.5 — Oversold — potential reversal
- **Distance from 52W High:** -30.7% — More than 20% off 52-week high

**Overall Technical Rating: Bearish** (score 0/5)

---

### 📰 News Intelligence
- **Company News Score:** 8.0/10 (raw keyword score: +3)
- **Market-Wide Sentiment:** Mildly Bullish (+3)

**Recent Headlines (10):**
  - SpaceX stock in focus: Why this strategist says to think like Woody — not Buzz Lightyear
  - Memory's price surge threatens Apple's 'magic formula' — and the economy: WSJ reporter
  - Microsoft Stock Gains on $4.7B AI Data Center Expansion in Wisconsin
  - What's Going on with Alphabet Stock Tuesday
  - The AI Trade Is Wobbling. Berkshire Hathaway Was Built for a Market Like This.
  - _(+ 5 more)_

---

### 🏛️ Macro Environment (FRED)
- **Fed Funds Rate:** 3.63% — Neutral (2–4%)
- **CPI Inflation (YoY):** 4.3% — Hot inflation (>4%)
- **Unemployment:** 4.3% — Softening (4–5.5%)
- **10Y-2Y Yield Spread:** 0.27% — Flat (0–0.5%)
- **VIX:** 17.3 — Moderate (15–25)
- **Real GDP Growth:** 1.6% — Moderate (1–3%)

**Macro Score: 4.2/10 — Macro Headwind**

---

### ⚠️ Risk Assessment
**Key Risks:**
  - Price below 200-day MA — long-term downtrend in place
  - Stock is 31% off its 52-week high

**Strengths:**
  - Strong ROE above 20%
  - Z-Score in safe zone

---

### 🏆 Portfolio Manager — Final Decision (4-Factor)

## 🔵 WATCH
**Composite Score: 5.4 / 10**  (Watch)

| Factor | Score | Weight | Contribution |
|--------|-------|--------|-------------|
| Fundamental | 7.9/10 | 40% | 3.14 |
| Technical   | 0.0/10 | 25%   | 0.00 |
| News        | 8.0/10 | 20%        | 1.60 |
| Macro       | 4.2/10 | 15%      | 0.62 |
| **Composite** | **5.4/10** | 100% | — |

**Guidance:** Above-average score but not fully confirmed. Add to watchlist and enter on improvement in the weakest factor.

---

# AAPL — Decision Report (2026-06-23)

---

### 📊 Fundamental Analysis
- **Return on Equity (ROE):** 151.9% — Strong (≥20%)
- **Debt-to-Equity (D/E):** 1.34 — Moderate (0.5–1.5)
- **Price-to-Earnings (P/E):** 39.0x — Stretched (30–50)
- **Piotroski F-Score:** 7/9 — Good (6–7)
- **Altman Z-Score:** 11.45 — Safe zone (≥3.0)

**Overall Fundamental Rating: Buy**

---

### 📈 Technical Analysis
- **Current Price:** $297.67
- **50-Day MA:** $290.11 — price is above (momentum positive)
- **200-Day MA:** $268.38 — price is above (long-term trend up)
- **MA Cross:** Golden Cross (bullish)
- **RSI (14):** 32.9 — Oversold — potential reversal
- **Distance from 52W High:** -5.6% — Within 20% of high

**Overall Technical Rating: Bullish** (score 4/5)

---

### 📰 News Intelligence
- **Company News Score:** 6.0/10 (raw keyword score: +1)
- **Market-Wide Sentiment:** Mildly Bullish (+3)

**Recent Headlines (10):**
  - Memory's price surge threatens Apple's 'magic formula' — and the economy: WSJ reporter
  - META Expands $299 Smart Glasses Lineup As It Challenges Apple’s AR Ambitions — Stock Gains Nearly 1.5%
  - Intel's Stock Is Soaring. Is It Too Late to Buy?
  - UK tribunal gives go ahead for $4 billion lawsuit against Apple over iCloud services
  - Tim Cook issues chilling warning as ‘hundred-year-flood’ strikes US tech giants like Apple, Dell, HP. Get protection now
  - _(+ 5 more)_

---

### 🏛️ Macro Environment (FRED)
- **Fed Funds Rate:** 3.63% — Neutral (2–4%)
- **CPI Inflation (YoY):** 4.3% — Hot inflation (>4%)
- **Unemployment:** 4.3% — Softening (4–5.5%)
- **10Y-2Y Yield Spread:** 0.27% — Flat (0–0.5%)
- **VIX:** 17.3 — Moderate (15–25)
- **Real GDP Growth:** 1.6% — Moderate (1–3%)

**Macro Score: 4.2/10 — Macro Headwind**

---

### ⚠️ Risk Assessment
**Key Risks:** No major red flags identified.

**Strengths:**
  - Strong ROE above 20%
  - Z-Score in safe zone
  - All technical indicators aligned bullishly

---

### 🏆 Portfolio Manager — Final Decision (4-Factor)

## 🟢 BUY
**Composite Score: 6.7 / 10**  (Buy)

| Factor | Score | Weight | Contribution |
|--------|-------|--------|-------------|
| Fundamental | 7.1/10 | 40% | 2.86 |
| Technical   | 8.0/10 | 25%   | 2.00 |
| News        | 6.0/10 | 20%        | 1.20 |
| Macro       | 4.2/10 | 15%      | 0.62 |
| **Composite** | **6.7/10** | 100% | — |

**Guidance:** Predominantly positive signal. Good entry opportunity with manageable risk. Monitor news flow for any deterioration.

...

---

# TSLA — Decision Report (2026-06-23)

---

### 📊 Fundamental Analysis
- **Return on Equity (ROE):** 4.6% — Weak (<10%)
- **Debt-to-Equity (D/E):** 0.18 — Low leverage (≤0.5)
- **Price-to-Earnings (P/E):** 378.0x — Expensive (>50)
- **Piotroski F-Score:** 6/9 — Good (6–7)
- **Altman Z-Score:** 17.20 — Safe zone (≥3.0)

**Overall Fundamental Rating: Buy**

---

### 📈 Technical Analysis
- **Current Price:** $381.87
- **50-Day MA:** $404.33 — price is below (momentum negative)
- **200-Day MA:** $417.54 — price is below (long-term trend down)
- **MA Cross:** Death Cross (bearish)
- **RSI (14):** 36.4 — Oversold — potential reversal
- **Distance from 52W High:** -22.0% — More than 20% off 52-week high

**Overall Technical Rating: Bearish** (score 0/5)

---

### 📰 News Intelligence
- **Company News Score:** 5.0/10 (raw keyword score: +0)
- **Market-Wide Sentiment:** Mildly Bullish (+3)

**Recent Headlines (10):**
  - Tesla pushes back as fatal Texas crash reignites self-driving scrutiny
  - SpaceX stock in focus: Why this strategist says to think like Woody — not Buzz Lightyear
  - Tesla Europe Registrations Jump 57% As Recovery Gains Momentum
  - Tesla Stock’s Good News on Sales Overwhelmed by the Bad
  - SPCX: Should you Buy the Dip?
  - _(+ 5 more)_

---

### 🏛️ Macro Environment (FRED)
- **Fed Funds Rate:** 3.63% — Neutral (2–4%)
- **CPI Inflation (YoY):** 4.3% — Hot inflation (>4%)
- **Unemployment:** 4.3% — Softening (4–5.5%)
- **10Y-2Y Yield Spread:** 0.27% — Flat (0–0.5%)
- **VIX:** 17.3 — Moderate (15–25)
- **Real GDP Growth:** 1.6% — Moderate (1–3%)

**Macro Score: 4.2/10 — Macro Headwind**

---

### ⚠️ Risk Assessment
**Key Risks:**
  - Elevated P/E of 378.0x — valuation highly sensitive to growth misses
  - Price below 200-day MA — long-term downtrend in place

**Strengths:**
  - Z-Score in safe zone

---

### 🏆 Portfolio Manager — Final Decision (4-Factor)

## 🟡 HOLD
**Composite Score: 3.9 / 10**  (Hold)

| Factor | Score | Weight | Contribution |
|--------|-------|--------|-------------|
| Fundamental | 5.7/10 | 40% | 2.29 |
| Technical   | 0.0/10 | 25%   | 0.00 |
| News        | 5.0/10 | 20%        | 1.00 |
| Macro       | 4.2/10 | 15%      | 0.62 |
| **Composite** | **3.9/10** | 100% | — |

**Guidance:** Mixed signals across factors. No clear catalyst in either direction. Reassess next quarter.

---

# JPM — Decision Report (2026-06-23)

---

### 📊 Fundamental Analysis
- **Return on Equity (ROE):** 15.7% — Adequate (10–20%)
- **Debt-to-Equity (D/E):** 1.38 — Moderate (0.5–1.5)
- **Price-to-Earnings (P/E):** 15.7x — Fair value (15–30)
- **Piotroski F-Score:** 2/9 — Poor (≤3)
- **Altman Z-Score:** n/a

**Overall Fundamental Rating: Hold**

---

### 📈 Technical Analysis
- **Current Price:** $334.41
- **50-Day MA:** $309.78 — price is above (momentum positive)
- **200-Day MA:** $305.54 — price is above (long-term trend up)
- **MA Cross:** Golden Cross (bullish)
- **RSI (14):** 76.8 — Overbought — caution
- **Distance from 52W High:** 0.0% — Near 52-week high (strong momentum)

**Overall Technical Rating: Neutral** (score 3/5)

---

### 📰 News Intelligence
- **Company News Score:** 5.0/10 (raw keyword score: +0)
- **Market-Wide Sentiment:** Mildly Bullish (+3)

**Recent Headlines (10):**
  - Wall Street Drags S&P 500 Targets Higher, Warning the Rally Is Riskier Than Ever
  - Why PAR Technology Corporation (PAR) Is Back on JPMorgan’s Radar
  - Is Upstream Bio, Inc. (UPB) the Best Low Priced Stocks to Get Rich in 2026?
  - After forcing workers back to the office, Goldman Sachs and JPMorgan Chase are now letting their staff work remotely—but only for the World Cup
  - 1 Mega-Cap Stock to Target This Week and 2 We Ignore
  - _(+ 5 more)_

---

### 🏛️ Macro Environment (FRED)
- **Fed Funds Rate:** 3.63% — Neutral (2–4%)
- **CPI Inflation (YoY):** 4.3% — Hot inflation (>4%)
- **Unemployment:** 4.3% — Softening (4–5.5%)
- **10Y-2Y Yield Spread:** 0.27% — Flat (0–0.5%)
- **VIX:** 17.3 — Moderate (15–25)
- **Real GDP Growth:** 1.6% — Moderate (1–3%)

**Macro Score: 4.2/10 — Macro Headwind**

---

### ⚠️ Risk Assessment
**Key Risks:**
  - Low F-Score (2/9) indicates deteriorating fundamentals
  - RSI 76.8 — overbought, short-term pullback likely

---

### 🏆 Portfolio Manager — Final Decision (4-Factor)

## 🟡 HOLD
**Composite Score: 4.8 / 10**  (Hold)

| Factor | Score | Weight | Contribution |
|--------|-------|--------|-------------|
| Fundamental | 4.3/10 | 40% | 1.71 |
| Technical   | 6.0/10 | 25%   | 1.50 |
| News        | 5.0/10 | 20%        | 1.00 |
| Macro       | 4.2/10 | 15%      | 0.62 |
| **Composite** | **4.8/10** | 100% | — |

**Guidance:** Mixed signals across factors. No clear catalyst in either direction. Reassess next quarter.

---

---
## Cell 12 — Save reports to disk (no edits needed)
---

In [47]:
SAVE_DIR = Path('reports') / ANALYSIS_DATE
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save screen CSV
screen_csv = SAVE_DIR / 'screen_results.csv'
screen_df.to_csv(screen_csv, index=False)
print(f'Screen saved  : {screen_csv}')

# Save composite scores CSV
comp_csv = SAVE_DIR / 'composite_scores.csv'
comp_df.to_csv(comp_csv, index=False)
print(f'Composite saved: {comp_csv}')

# Save macro snapshot
macro_path = SAVE_DIR / 'macro_snapshot.txt'
macro_path.write_text(f"Date: {ANALYSIS_DATE}\nMacro Score: {macro.macro_score:.1f}/10 ({macro.macro_label})\n{macro.macro_summary}\n", encoding='utf-8')
print(f'Macro saved   : {macro_path}')

# Save decision reports
for ticker, report in decision_reports.items():
    report_path = SAVE_DIR / f'{ticker}_decision.md'
    report_path.write_text(report, encoding='utf-8')
    print(f'Report saved  : {report_path}')

print('\nAll files saved.')

Screen saved  : reports/2026-06-23/screen_results.csv
Composite saved: reports/2026-06-23/composite_scores.csv
Macro saved   : reports/2026-06-23/macro_snapshot.txt
Report saved  : reports/2026-06-23/META_decision.md
Report saved  : reports/2026-06-23/MSFT_decision.md
Report saved  : reports/2026-06-23/AAPL_decision.md
Report saved  : reports/2026-06-23/AMZN_decision.md
Report saved  : reports/2026-06-23/GOOGL_decision.md
Report saved  : reports/2026-06-23/NVDA_decision.md
Report saved  : reports/2026-06-23/TSLA_decision.md
Report saved  : reports/2026-06-23/JPM_decision.md

All files saved.


---
## Cell 13 — Summary Dashboard (no edits needed)
---

In [48]:
from IPython.display import Markdown, display as ipy_display

summary_lines = [
    f'# Trading Decision Maker — Summary ({ANALYSIS_DATE})',
    f'**Screened:** {len(TICKERS)} tickers  |  '
    f'**Passed screen:** {len(passed_tickers)}  |  '
    f'**Top Picks (composite ≥6.0):** {len([t for t,cs in composite_scores.items() if cs.recommendation in ("Strong Buy","Buy")])}\n',

    '## Macro & Market Context',
    f'| Indicator | Value |',
    f'|-----------|-------|',
    f'| Market News Sentiment | **{market_sentiment}** ({market_raw_score:+d}) |',
    f'| Macro Environment     | **{macro.macro_label}** ({macro.macro_score:.1f}/10) |',
    f'| {macro.macro_summary} | |',
    '',
    '## Composite Scores',
    '| Ticker | Fund | Tech | News | Macro | **Composite** | Action |',
    '|--------|------|------|------|-------|--------------|--------|',
]

for f, t in rows:
    cs = composite_scores[f.ticker]
    summary_lines.append(
        f'| {f.ticker} | {cs.fund_score_10:.1f} | {cs.tech_score_10:.1f} | {cs.news_score_10:.1f} | {cs.macro_score_10:.1f} | **{cs.composite:.1f}** | {cs.action} |'
    )

ipy_display(Markdown('\n'.join(summary_lines)))

# Trading Decision Maker — Summary (2026-06-23)
**Screened:** 8 tickers  |  **Passed screen:** 0  |  **Top Picks (composite ≥6.0):** 4

## Macro & Market Context
| Indicator | Value |
|-----------|-------|
| Market News Sentiment | **Mildly Bullish** (+3) |
| Macro Environment     | **Macro Headwind** (4.2/10) |
| Fed Funds 3.63% | CPI YoY 4.3% | Unemployment 4.3% | 10Y-2Y Spread 0.27% | VIX 17.3 | GDP Growth 1.6% | |

## Composite Scores
| Ticker | Fund | Tech | News | Macro | **Composite** | Action |
|--------|------|------|------|-------|--------------|--------|
| META | 7.9 | 2.0 | 6.0 | 4.2 | **5.5** | 🔵 WATCH |
| MSFT | 7.9 | 0.0 | 8.0 | 4.2 | **5.4** | 🔵 WATCH |
| AAPL | 7.1 | 8.0 | 6.0 | 4.2 | **6.7** | 🟢 BUY |
| AMZN | 7.1 | 6.0 | 6.0 | 4.2 | **6.2** | 🟢 BUY |
| GOOGL | 6.4 | 8.0 | 6.0 | 4.2 | **6.4** | 🟢 BUY |
| NVDA | 6.4 | 6.0 | 7.0 | 4.2 | **6.1** | 🟢 BUY |
| TSLA | 5.7 | 0.0 | 5.0 | 4.2 | **3.9** | 🟡 HOLD |
| JPM | 4.3 | 6.0 | 5.0 | 4.2 | **4.8** | 🟡 HOLD |

---
## Cell 14 — 🤖 Ollama AI Narrative Layer — Setup & Check
Generates a one-page written investment summary per ticker using a **local LLM** (no API key, no cost, fully private).

**Prerequisites — run once before this cell:**
```bash
# 1. Install Ollama: https://ollama.com/download
# 2. Pull the model (choose one):
#    ollama pull llama3          # ~4.7GB — recommended
#    ollama pull mistral         # ~4.1GB — faster, slightly less capable
#    ollama pull phi3            # ~2.3GB — lightest option
# 3. Ollama runs as a background service automatically after install
```
> **This cell is optional.** If Ollama is not installed or the model is not pulled,
> the notebook continues normally — Ollama reports are simply skipped.

---

In [49]:
import subprocess
import importlib

# ─── CONFIGURATION — set your preferred model here ───────────────────────────
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  OLLAMA MODEL                                                           ║
# ║  Options: "llama3"  "mistral"  "phi3"  "gemma"                         ║
# ║  Must match a model you have pulled via: ollama pull <model>           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
OLLAMA_MODEL = "llama3"

# ─── Check Ollama is available ────────────────────────────────────────────────
OLLAMA_AVAILABLE = False

try:
    import ollama as _ollama_module
    OLLAMA_AVAILABLE = True
    print("ollama Python package  : ✓ installed")
except ImportError:
    print("ollama Python package  : ✗ not installed  →  run: pip install ollama")

if OLLAMA_AVAILABLE:
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            print("Ollama service         : ✓ running")
            available_models = result.stdout.strip()
            print(f"\nPulled models:\n{available_models}")
            if OLLAMA_MODEL in result.stdout:
                print(f"\nModel '{OLLAMA_MODEL}' : ✓ ready")
            else:
                print(f"\nModel '{OLLAMA_MODEL}' : ✗ not found")
                print(f"  → Run in terminal:  ollama pull {OLLAMA_MODEL}")
                OLLAMA_AVAILABLE = False
        else:
            print("Ollama service         : ✗ not running")
            print("  → Start Ollama: open the Ollama app, or run 'ollama serve' in terminal")
            OLLAMA_AVAILABLE = False
    except (FileNotFoundError, subprocess.TimeoutExpired):
        print("Ollama service         : ✗ not found or timed out")
        print("  → Install from: https://ollama.com/download")
        OLLAMA_AVAILABLE = False

if OLLAMA_AVAILABLE:
    print(f"\n✅ Ready to generate AI narratives using '{OLLAMA_MODEL}'")
else:
    print("\n⚠️  Ollama not available — AI narrative cell will be skipped gracefully.")
    print("   The rest of the notebook (screens, scores, reports) is unaffected.")

ollama Python package  : ✓ installed
Ollama service         : ✓ running

Pulled models:
NAME             ID              SIZE      MODIFIED      
llama3:latest    365c0bd3c000    4.7 GB    3 minutes ago

Model 'llama3' : ✓ ready

✅ Ready to generate AI narratives using 'llama3'


---
## Cell 15 — 🤖 Ollama AI Investment Narratives
Feeds each ticker's composite scores, key metrics, and recent headlines into the local LLM
and generates a structured one-page investment summary.

> **Output per ticker:** Business Quality · Financial Strength · Risks · Growth Opportunities · Valuation · Final Recommendation

---

In [51]:
from IPython.display import Markdown, display as ipy_display

# ─── Prompt builder ───────────────────────────────────────────────────────────
def build_ollama_prompt(f: FundamentalMetrics, t, cs: CompositeScore, headlines: list[str]) -> str:

    def sfmt(val, fmt):
        """Safely format a value that may be None."""
        return "n/a" if val is None else format(val, fmt)

    # Technical block
    if t:
        tech_block = (
            f"Price: ${sfmt(t.price, '.2f')} | 50DMA: ${sfmt(t.ma_50, '.2f')} | 200DMA: ${sfmt(t.ma_200, '.2f')} | "
            f"RSI: {sfmt(t.rsi_14, '.1f')} | Technical Rating: {t.technical_rating}"
        )
    else:
        tech_block = "Technical data unavailable"

    # News block
    news_block = "\n".join(f"  - {h}" for h in headlines[:5]) if headlines else "  No headlines available"

    # Macro block
    macro_parts = []
    if macro.fed_funds_rate is not None: macro_parts.append(f"Fed Rate: {macro.fed_funds_rate:.2f}%")
    if macro.cpi_yoy        is not None: macro_parts.append(f"CPI YoY: {macro.cpi_yoy:.1f}%")
    if macro.unemployment   is not None: macro_parts.append(f"Unemployment: {macro.unemployment:.1f}%")
    if macro.yield_spread   is not None: macro_parts.append(f"10Y-2Y Spread: {macro.yield_spread:.2f}%")
    if macro.vix            is not None: macro_parts.append(f"VIX: {macro.vix:.1f}")
    macro_block = " | ".join(macro_parts) if macro_parts else "Macro data unavailable"

    return f"""You are a senior equity research analyst. Write a concise, structured one-page investment summary.

STOCK: {f.ticker}
DATE: {ANALYSIS_DATE}

=== FUNDAMENTAL DATA ===
ROE: {sfmt(f.roe, '.1%')} | D/E: {sfmt(f.debt_to_equity, '.2f')} | P/E: {sfmt(f.pe_ratio, '.1f')}x
Piotroski F-Score: {f.piotroski_f_score}/9 | Altman Z-Score: {sfmt(f.altman_z_score, '.2f')}
Fundamental Rating: {f.fundamental_rating}

=== TECHNICAL DATA ===
{tech_block}

=== NEWS SENTIMENT ===
Company News Score: {cs.news_score_10:.1f}/10
Market Sentiment: {market_sentiment} ({market_raw_score:+d})
Recent Headlines:
{news_block}

=== MACRO ENVIRONMENT ===
{macro_block}
Macro Score: {macro.macro_score:.1f}/10 ({macro.macro_label})

=== COMPOSITE SCORE ===
Fundamental: {cs.fund_score_10:.1f}/10 (40%) | Technical: {cs.tech_score_10:.1f}/10 (25%)
News: {cs.news_score_10:.1f}/10 (20%) | Macro: {cs.macro_score_10:.1f}/10 (15%)
COMPOSITE: {cs.composite:.1f}/10 → {cs.recommendation}

Write exactly these six sections, each 2-4 sentences. Be direct, specific, and analytical:

1. BUSINESS QUALITY
2. FINANCIAL STRENGTH
3. KEY RISKS
4. GROWTH OPPORTUNITIES
5. VALUATION ASSESSMENT
6. FINAL RECOMMENDATION

End with one clear action sentence: Buy / Watch / Hold / Avoid — and why.
Do not include disclaimers or preamble. Start directly with section 1."""

# ─── Generate narratives ──────────────────────────────────────────────────────
ollama_reports: dict[str, str] = {}

if not OLLAMA_AVAILABLE:
    print("⚠️  Ollama not available — skipping AI narrative generation.")
    print("   All other reports (rule-based decision reports, composite scores) are still available.")
else:
    import ollama as _ollama

    print(f"Generating AI narratives using '{OLLAMA_MODEL}' ...")
    print(f"(Expect ~15–60 seconds per ticker depending on your hardware)\n")

    for f, t in rows:
        cs        = composite_scores[f.ticker]
        headlines = company_news_map.get(f.ticker, [])
        prompt    = build_ollama_prompt(f, t, cs, headlines)

        print(f"  {f.ticker:<6} generating ...", end="", flush=True)
        try:
            response = _ollama.chat(
                model=OLLAMA_MODEL,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0.3}   # low temp = more consistent, analytical output
            )
            narrative = response["message"]["content"].strip()
            ollama_reports[f.ticker] = narrative
            print(" done ✓")
        except Exception as e:
            ollama_reports[f.ticker] = f"_AI narrative failed: {e}_"
            print(f" FAILED ({e})")

    print(f"\nAI narratives complete: {len(ollama_reports)} tickers")

# ─── Display AI narratives ────────────────────────────────────────────────────
if ollama_reports:
    print("\n" + "═"*60)
    print(" AI INVESTMENT NARRATIVES")
    print("═"*60)
    for ticker, narrative in ollama_reports.items():
        cs = composite_scores[ticker]
        header = f"""# 🤖 {ticker} — AI Investment Narrative ({ANALYSIS_DATE})
**Composite Score: {cs.composite:.1f}/10 → {cs.action}**
*Generated by {OLLAMA_MODEL} (local, private)*

---
"""
        ipy_display(Markdown(header + narrative))
        ipy_display(Markdown("---"))

Generating AI narratives using 'llama3' ...
(Expect ~15–60 seconds per ticker depending on your hardware)

  META   generating ... done ✓
  MSFT   generating ... done ✓
  AAPL   generating ... done ✓
  AMZN   generating ... done ✓
  GOOGL  generating ... done ✓
  NVDA   generating ... done ✓
  TSLA   generating ... done ✓
  JPM    generating ... done ✓

AI narratives complete: 8 tickers

════════════════════════════════════════════════════════════
 AI INVESTMENT NARRATIVES
════════════════════════════════════════════════════════════


# 🤖 META — AI Investment Narrative (2026-06-23)
**Composite Score: 5.5/10 → 🔵 WATCH**
*Generated by llama3 (local, private)*

---
**BUSINESS QUALITY**
Meta's strong ROE of 27.8% and low D/E ratio of 0.39 indicate a financially healthy company with high profitability. The Piotroski F-Score of 4/9 suggests some concerns about earnings quality, but the Altman Z-Score of 7.01 indicates low bankruptcy risk.

**FINANCIAL STRENGTH**
Meta's financials are robust, with a strong balance sheet and manageable debt levels. This provides a solid foundation for future growth initiatives.

**KEY RISKS**
The company faces risks from increasing competition in the AI-powered glasses market and potential regulatory challenges related to its prediction markets app Arena. Additionally, the macro environment remains uncertain, with inflation concerns and interest rate hikes potentially impacting consumer spending.

**GROWTH OPPORTUNITIES**
Meta's recent debut of AI-powered Meta Glasses offers a new revenue stream, while its prediction markets app Arena has the potential to disrupt traditional sports betting. The company's focus on AI-driven innovation positions it well for long-term growth.

**VALUATION ASSESSMENT**
At $564.16, Meta trades at 20.5x P/E, which is relatively high compared to historical averages. However, considering its strong financials and growth opportunities, the stock may be justified in trading at a premium valuation.

**FINAL RECOMMENDATION**
Based on our composite score of 5.5/10, we recommend **WATCH** for now. While Meta's fundamentals are strong, the technical picture is bearish, and macro headwinds remain a concern. We will continue to monitor the company's progress and adjust our recommendation accordingly.

---

# 🤖 MSFT — AI Investment Narrative (2026-06-23)
**Composite Score: 5.4/10 → 🔵 WATCH**
*Generated by llama3 (local, private)*

---
**MSFT - Microsoft Corporation**

**1. BUSINESS QUALITY**
Microsoft's strong business quality is reflected in its high ROE of 29.6%, indicating efficient capital allocation. The company's low debt-to-equity ratio of 0.18 and Piotroski F-Score of 5/9 suggest a stable financial profile.

**2. FINANCIAL STRENGTH**
Microsoft's solid financial strength is demonstrated by its high Altman Z-Score of 7.80, indicating a low risk of bankruptcy. The company's fundamental rating is High Buy, further supporting its financial robustness.

**3. KEY RISKS**
The main risks facing Microsoft are macroeconomic headwinds, including the rising interest rates and inflationary pressures. The company's reliance on cloud computing and artificial intelligence (AI) also exposes it to potential disruptions in these areas.

**4. GROWTH OPPORTUNITIES**
Microsoft's growth opportunities lie in its expanding AI capabilities, particularly with its $4.7 billion AI data center expansion in Wisconsin. The company's strong presence in the gaming industry through Xbox also presents a promising area for growth.

**5. VALUATION ASSESSMENT**
Microsoft's P/E ratio of 27.2x is slightly above its historical average, indicating moderate valuation. However, the company's technical rating is bearish due to its recent price decline and RSI reading of 16.5.

**6. FINAL RECOMMENDATION**
Based on our composite score, we recommend a **Watch** position for MSFT, as the stock's fundamental strength is offset by macroeconomic headwinds and moderate valuation. We will closely monitor the company's performance and market conditions before making a buy or sell decision.

---

...

---

# 🤖 TSLA — AI Investment Narrative (2026-06-23)
**Composite Score: 3.9/10 → 🟡 HOLD**
*Generated by llama3 (local, private)*

---
**BUSINESS QUALITY**
TSLA's ROE of 4.6% suggests a stable business, while its Piotroski F-Score of 6/9 indicates a decent track record of profitability. However, the company's high debt-to-equity ratio (0.18) and low return on equity raise concerns about its financial leverage.

**FINANCIAL STRENGTH**
TSLA's Altman Z-Score of 17.20 suggests a strong ability to meet its short-term obligations. The company's fundamental rating is "Buy", indicating a stable financial position.

**KEY RISKS**
The recent fatal Texas crash has reignited scrutiny over TSLA's self-driving technology, posing a significant reputational risk. Additionally, the company's high valuation (P/E of 378.0x) makes it vulnerable to market fluctuations.

**GROWTH OPPORTUNITIES**
TSLA's Europe registrations jumped 57% in recent months, indicating a strong recovery momentum. The company's expanding product offerings and growing demand for electric vehicles also present opportunities for growth.

**VALUATION ASSESSMENT**
TSLA's valuation is rich, with a P/E ratio significantly higher than its peers. While the company's growth prospects are promising, the current price may not fully reflect these expectations.

**FINAL RECOMMENDATION**
Based on our composite score of 3.9/10, we recommend HOLDING TSLA shares. The company's stable business and strong financial position are offset by concerns over its valuation and reputational risks.

---

# 🤖 JPM — AI Investment Narrative (2026-06-23)
**Composite Score: 4.8/10 → 🟡 HOLD**
*Generated by llama3 (local, private)*

---
**JPM - Investment Summary**

**1. BUSINESS QUALITY**
JPMorgan Chase's (JPM) business quality is solid, with a strong track record of generating returns on equity (ROE) at 15.7%. However, the company's financial leverage ratio (D/E) stands at 1.38, indicating some reliance on debt financing.

**2. FINANCIAL STRENGTH**
Financially speaking, JPM's Piotroski F-Score is low at 2/9, suggesting limited profitability and cash flow generation. The Altman Z-Score is not applicable due to the company's financial leverage. Our Fundamental Rating is Hold.

**3. KEY RISKS**
Key risks include macroeconomic headwinds, with a Macro Score of 4.2/10 due to rising inflation (CPI YoY: 4.3%) and interest rates (Fed Rate: 3.63%). Additionally, the company's reliance on debt financing may amplify these risks.

**4. GROWTH OPPORTUNITIES**
Growth opportunities are limited, with a Technical Rating of Neutral and no significant catalysts driving growth.

**5. VALUATION ASSESSMENT**
Valuation-wise, JPM's price-to-earnings (P/E) ratio stands at 15.7x, which is slightly above the industry average. The company's recent stock performance has been strong, with a RSI of 76.8 indicating some overbought conditions.

**6. FINAL RECOMMENDATION**
Based on our Composite Score of 4.8/10, we recommend HOLDING JPM. While the company's business quality is solid, macroeconomic headwinds and valuation concerns temper our enthusiasm for the stock.

**ACTION:** Hold

---

---
## Cell 16 — Save AI Narratives to disk (no edits needed)
---

In [52]:
# Save Ollama AI narrative reports to the same reports directory
if ollama_reports:
    for ticker, narrative in ollama_reports.items():
        ai_path = SAVE_DIR / f'{ticker}_ai_narrative.md'
        cs = composite_scores[ticker]
        full_text = (
            f"# {ticker} — AI Investment Narrative ({ANALYSIS_DATE})\n"
            f"**Model:** {OLLAMA_MODEL}  |  "
            f"**Composite Score:** {cs.composite:.1f}/10 → {cs.recommendation}\n\n"
            f"---\n\n"
            f"{narrative}\n"
        )
        ai_path.write_text(full_text, encoding='utf-8')
        print(f'AI narrative saved : {ai_path}')
    print('\nAll AI narratives saved.')
else:
    print('No AI narratives to save (Ollama was not available or was skipped).')

AI narrative saved : reports/2026-06-23/META_ai_narrative.md
AI narrative saved : reports/2026-06-23/MSFT_ai_narrative.md
AI narrative saved : reports/2026-06-23/AAPL_ai_narrative.md
AI narrative saved : reports/2026-06-23/AMZN_ai_narrative.md
AI narrative saved : reports/2026-06-23/GOOGL_ai_narrative.md
AI narrative saved : reports/2026-06-23/NVDA_ai_narrative.md
AI narrative saved : reports/2026-06-23/TSLA_ai_narrative.md
AI narrative saved : reports/2026-06-23/JPM_ai_narrative.md

All AI narratives saved.
